In [17]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V8
================================================

KEY INNOVATION: Leiden Clustering for Region-Specific Comparison

Instead of comparing whole tissue patterns, we:
1. Use Leiden clustering (resolution=0.1) to divide tissue into regions
2. Compute signatures for each region separately
3. Match gene and m/z patterns WITHIN corresponding regions
4. Use top 10% importance within each region

Benefits:
- Region-specific: Compares hippocampus to hippocampus, cortex to cortex
- Robust to misalignment: Regional patterns match even with imperfect alignment
- Biologically meaningful: Captures region-specific gene-metabolite relationships
- Natural landmarks: Clusters provide alignment anchors

Parameters:
- RNA: 55 μm pixel size, hexagonal grid, 6 neighbors
- MSI: 60 μm pixel size, Cartesian grid, 8 neighbors
- Leiden resolution: 0.1 (yields ~5-10 major regions)
- RNA rotated 180° to align with MSI

Author: V8 - Region-Specific Matching with Leiden Clustering
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from scipy.interpolate import griddata
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from typing import List, Dict, Tuple, Optional
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
import pickle

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

RNA_PIXEL_SIZE = 55  # μm (Visium)
MSI_PIXEL_SIZE = 60  # μm
N_CLUSTERS = 3  # Fixed number of clusters per sample

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/gene_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]


AAD_TARGET_GENES = [
    'Thy1', 'App', 'Apoe', 'Mapt'
]


In [18]:

# =============================================================================
# LEIDEN CLUSTERING FOR SPATIAL REGIONS
# =============================================================================

def compute_spatial_leiden_fixed_clusters(
    adata: sc.AnnData,
    n_clusters: int = 3,
    n_neighbors: int = 15,
    use_rep: str = None
) -> np.ndarray:
    """
    Compute Leiden clustering with exactly n_clusters.
    Uses binary search on resolution to achieve target cluster count.
    
    Args:
        adata: AnnData object with spatial coordinates in obs['x_um'], obs['y_um']
        n_clusters: Target number of clusters (default: 3)
        n_neighbors: Number of neighbors for graph
        use_rep: Representation to use ('X_pca', None for raw)
    
    Returns:
        Cluster labels as numpy array
    """
    # Make a copy to avoid modifying original
    adata_temp = adata.copy()
    
    # Ensure we have spatial coordinates
    if 'x_um' in adata_temp.obs and 'y_um' in adata_temp.obs:
        adata_temp.obsm['spatial'] = np.column_stack([
            adata_temp.obs['x_um'].values,
            adata_temp.obs['y_um'].values
        ])
    
    # Preprocessing if needed
    if use_rep is None:
        # Use expression data
        sc.pp.normalize_total(adata_temp, target_sum=1e4)
        sc.pp.log1p(adata_temp)
        sc.pp.highly_variable_genes(adata_temp, n_top_genes=min(500, adata_temp.n_vars))
        sc.pp.pca(adata_temp, n_comps=min(30, adata_temp.n_obs - 1, adata_temp.n_vars - 1))
        use_rep = 'X_pca'
    
    # Build neighbor graph
    sc.pp.neighbors(adata_temp, n_neighbors=n_neighbors, use_rep=use_rep)
    
    # Binary search for resolution that gives n_clusters
    res_low, res_high = 0.01, 2.0
    best_resolution = 0.1
    best_diff = float('inf')
    
    for _ in range(20):  # Max 20 iterations
        res_mid = (res_low + res_high) / 2
        sc.tl.leiden(adata_temp, resolution=res_mid, key_added='leiden')
        current_clusters = len(adata_temp.obs['leiden'].unique())
        
        diff = abs(current_clusters - n_clusters)
        if diff < best_diff:
            best_diff = diff
            best_resolution = res_mid
        
        if current_clusters == n_clusters:
            break
        elif current_clusters < n_clusters:
            res_low = res_mid
        else:
            res_high = res_mid
    
    # Final clustering with best resolution
    sc.tl.leiden(adata_temp, resolution=best_resolution, key_added='leiden')
    labels = adata_temp.obs['leiden'].values.astype(int)
    
    # If still not exactly n_clusters, merge or split
    unique_labels = np.unique(labels)
    if len(unique_labels) > n_clusters:
        # Merge smallest clusters
        labels = _merge_clusters(labels, adata_temp.obsm['spatial'] if 'spatial' in adata_temp.obsm else None, n_clusters)
    elif len(unique_labels) < n_clusters:
        # Split largest cluster
        labels = _split_clusters(labels, adata_temp.obsm['spatial'] if 'spatial' in adata_temp.obsm else None, n_clusters)
    
    return labels


def _merge_clusters(labels: np.ndarray, coords: np.ndarray, n_clusters: int) -> np.ndarray:
    """Merge smallest clusters until we have n_clusters"""
    labels = labels.copy()
    
    while len(np.unique(labels)) > n_clusters:
        unique, counts = np.unique(labels, return_counts=True)
        
        # Find smallest cluster
        smallest_idx = np.argmin(counts)
        smallest_label = unique[smallest_idx]
        
        # Find nearest cluster by centroid
        if coords is not None:
            centroids = {l: coords[labels == l].mean(axis=0) for l in unique}
            smallest_centroid = centroids[smallest_label]
            
            min_dist = float('inf')
            nearest_label = None
            for l in unique:
                if l != smallest_label:
                    dist = np.linalg.norm(centroids[l] - smallest_centroid)
                    if dist < min_dist:
                        min_dist = dist
                        nearest_label = l
        else:
            # Merge with largest cluster
            nearest_label = unique[np.argmax(counts)]
            if nearest_label == smallest_label:
                nearest_label = unique[np.argsort(counts)[-2]]
        
        # Merge
        labels[labels == smallest_label] = nearest_label
    
    # Relabel to 0, 1, 2, ...
    unique = np.unique(labels)
    label_map = {old: new for new, old in enumerate(unique)}
    labels = np.array([label_map[l] for l in labels])
    
    return labels


def _split_clusters(labels: np.ndarray, coords: np.ndarray, n_clusters: int) -> np.ndarray:
    """Split largest cluster until we have n_clusters"""
    labels = labels.copy()
    
    while len(np.unique(labels)) < n_clusters:
        unique, counts = np.unique(labels, return_counts=True)
        
        # Find largest cluster
        largest_idx = np.argmax(counts)
        largest_label = unique[largest_idx]
        
        # Split by k-means on coordinates
        if coords is not None:
            mask = labels == largest_label
            cluster_coords = coords[mask]
            
            # Simple k-means with k=2
            from sklearn.cluster import KMeans
            kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
            sub_labels = kmeans.fit_predict(cluster_coords)
            
            # Assign new label to one half
            new_label = labels.max() + 1
            mask_indices = np.where(mask)[0]
            labels[mask_indices[sub_labels == 1]] = new_label
        else:
            # Split in half by index
            mask = labels == largest_label
            mask_indices = np.where(mask)[0]
            mid = len(mask_indices) // 2
            new_label = labels.max() + 1
            labels[mask_indices[mid:]] = new_label
    
    # Relabel to 0, 1, 2, ...
    unique = np.unique(labels)
    label_map = {old: new for new, old in enumerate(unique)}
    labels = np.array([label_map[l] for l in labels])
    
    return labels


def compute_spatial_only_leiden_fixed(
    coords: np.ndarray,
    n_clusters: int = 3,
    n_neighbors: int = 15
) -> np.ndarray:
    """
    Compute Leiden clustering based on spatial coordinates only.
    Ensures exactly n_clusters.
    
    Args:
        coords: [N, 2] array of spatial coordinates
        n_clusters: Target number of clusters (default: 3)
        n_neighbors: Number of neighbors
    
    Returns:
        Cluster labels
    """
    import igraph as ig
    import leidenalg
    
    # Build k-NN graph
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    # Create edges with distance-based weights
    edges = []
    weights = []
    
    for i in range(len(coords)):
        for j_idx in range(1, n_neighbors + 1):
            j = indices[i, j_idx]
            dist = distances[i, j_idx]
            weight = 1.0 / (dist + 1e-8)
            edges.append((i, j))
            weights.append(weight)
    
    # Create igraph
    g = ig.Graph(n=len(coords), edges=edges, directed=False)
    g.es['weight'] = weights
    
    # Binary search for resolution
    res_low, res_high = 0.001, 5.0
    best_resolution = 0.1
    best_diff = float('inf')
    
    for _ in range(20):
        res_mid = (res_low + res_high) / 2
        partition = leidenalg.find_partition(
            g, 
            leidenalg.RBConfigurationVertexPartition,
            weights='weight',
            resolution_parameter=res_mid
        )
        current_clusters = len(set(partition.membership))
        
        diff = abs(current_clusters - n_clusters)
        if diff < best_diff:
            best_diff = diff
            best_resolution = res_mid
        
        if current_clusters == n_clusters:
            break
        elif current_clusters < n_clusters:
            res_low = res_mid
        else:
            res_high = res_mid
    
    # Final clustering
    partition = leidenalg.find_partition(
        g, 
        leidenalg.RBConfigurationVertexPartition,
        weights='weight',
        resolution_parameter=best_resolution
    )
    labels = np.array(partition.membership)
    
    # Ensure exactly n_clusters
    if len(np.unique(labels)) > n_clusters:
        labels = _merge_clusters(labels, coords, n_clusters)
    elif len(np.unique(labels)) < n_clusters:
        labels = _split_clusters(labels, coords, n_clusters)
    
    return labels


def match_regions_by_centroid(
    gene_coords: np.ndarray,
    gene_clusters: np.ndarray,
    mz_coords: np.ndarray,
    mz_clusters: np.ndarray
) -> Dict[int, int]:
    """
    Match gene regions to m/z regions based on centroid proximity.
    
    Returns:
        Dictionary mapping gene_cluster_id -> mz_cluster_id
    """
    # Compute centroids for each cluster
    gene_centroids = {}
    for c in np.unique(gene_clusters):
        mask = gene_clusters == c
        gene_centroids[c] = gene_coords[mask].mean(axis=0)
    
    mz_centroids = {}
    for c in np.unique(mz_clusters):
        mask = mz_clusters == c
        mz_centroids[c] = mz_coords[mask].mean(axis=0)
    
    # Match by nearest centroid
    gene_cluster_ids = list(gene_centroids.keys())
    mz_cluster_ids = list(mz_centroids.keys())
    
    gene_centroid_array = np.array([gene_centroids[c] for c in gene_cluster_ids])
    mz_centroid_array = np.array([mz_centroids[c] for c in mz_cluster_ids])
    
    # Compute distance matrix
    dist_matrix = cdist(gene_centroid_array, mz_centroid_array)
    
    # Greedy matching (Hungarian algorithm would be better but this is simpler)
    mapping = {}
    used_mz = set()
    
    for gene_idx in range(len(gene_cluster_ids)):
        min_dist = np.inf
        best_mz_idx = -1
        
        for mz_idx in range(len(mz_cluster_ids)):
            if mz_idx not in used_mz and dist_matrix[gene_idx, mz_idx] < min_dist:
                min_dist = dist_matrix[gene_idx, mz_idx]
                best_mz_idx = mz_idx
        
        if best_mz_idx >= 0:
            mapping[gene_cluster_ids[gene_idx]] = mz_cluster_ids[best_mz_idx]
            used_mz.add(best_mz_idx)
    
    return mapping


# =============================================================================
# COORDINATE TRANSFORMATION
# =============================================================================

def rotate_180(coords: np.ndarray) -> np.ndarray:
    """Rotate coordinates 180° around centroid"""
    center = coords.mean(axis=0)
    return 2 * center - coords


def align_rna_to_msi(rna_coords: np.ndarray, msi_coords: np.ndarray) -> np.ndarray:
    """Align RNA coordinates to MSI coordinate system"""
    rotated = rotate_180(rna_coords)
    
    rna_min, rna_max = rotated.min(axis=0), rotated.max(axis=0)
    msi_min, msi_max = msi_coords.min(axis=0), msi_coords.max(axis=0)
    
    rna_range = rna_max - rna_min
    msi_range = msi_max - msi_min
    
    scale = msi_range / (rna_range + 1e-8)
    aligned = (rotated - rna_min) * scale + msi_min
    
    return aligned


# =============================================================================
# REGION-SPECIFIC STATISTICS
# =============================================================================

@dataclass
class RegionStats:
    """Statistics for a single region (full region, not top 10%)"""
    cluster_id: int
    n_points: int
    centroid: np.ndarray
    mean_value: float
    std_value: float
    max_value: float
    min_value: float
    total_value: float
    median_value: float
    
    # Spatial extent
    x_range: Tuple[float, float]
    y_range: Tuple[float, float]
    area: float  # Approximate area
    
    # Distribution
    value_histogram: np.ndarray
    percentiles: np.ndarray  # [10, 25, 50, 75, 90]
    
    # Spatial distribution within region
    spatial_spread: float  # Std of distances from centroid
    value_weighted_centroid: np.ndarray  # Centroid weighted by values
    
    # Raw data for this region
    coords: np.ndarray = None
    values: np.ndarray = None
    importance: np.ndarray = None


def compute_region_stats(
    coords: np.ndarray,
    values: np.ndarray,
    importance: np.ndarray,
    cluster_labels: np.ndarray,
    n_hist_bins: int = 20
) -> Dict[int, RegionStats]:
    """
    Compute statistics for each region (FULL region, not top 10%).
    
    Args:
        coords: [N, 2] coordinates
        values: [N] feature values
        importance: [N] importance scores
        cluster_labels: [N] cluster assignments
        n_hist_bins: Number of histogram bins
    
    Returns:
        Dictionary mapping cluster_id -> RegionStats
    """
    region_stats = {}
    
    for cluster_id in np.unique(cluster_labels):
        mask = cluster_labels == cluster_id
        
        if mask.sum() < 5:
            continue
        
        region_coords = coords[mask]
        region_values = values[mask]
        region_importance = importance[mask]
        
        # Centroid
        centroid = region_coords.mean(axis=0)
        
        # Value-weighted centroid
        if region_values.sum() > 0:
            weights = region_values / region_values.sum()
            value_weighted_centroid = (region_coords * weights[:, np.newaxis]).sum(axis=0)
        else:
            value_weighted_centroid = centroid
        
        # Spatial spread (std of distances from centroid)
        distances = np.linalg.norm(region_coords - centroid, axis=1)
        spatial_spread = distances.std()
        
        # Approximate area (bounding box)
        x_range = (region_coords[:, 0].min(), region_coords[:, 0].max())
        y_range = (region_coords[:, 1].min(), region_coords[:, 1].max())
        area = (x_range[1] - x_range[0]) * (y_range[1] - y_range[0])
        
        # Value histogram (normalized)
        hist, _ = np.histogram(region_values, bins=n_hist_bins, density=True)
        hist = hist / (hist.sum() + 1e-8)
        
        # Percentiles
        percentiles = np.percentile(region_values, [10, 25, 50, 75, 90])
        
        region_stats[cluster_id] = RegionStats(
            cluster_id=cluster_id,
            n_points=mask.sum(),
            centroid=centroid,
            mean_value=region_values.mean(),
            std_value=region_values.std(),
            max_value=region_values.max(),
            min_value=region_values.min(),
            total_value=region_values.sum(),
            median_value=np.median(region_values),
            x_range=x_range,
            y_range=y_range,
            area=area,
            value_histogram=hist,
            percentiles=percentiles,
            spatial_spread=spatial_spread,
            value_weighted_centroid=value_weighted_centroid,
            coords=region_coords,
            values=region_values,
            importance=region_importance
        )
    
    return region_stats


# =============================================================================
# SPATIAL SIGNATURE WITH REGIONS
# =============================================================================

@dataclass
class SpatialSignature:
    """Spatial signature with region-specific information"""
    sample_id: str
    feature_name: str
    feature_type: str
    
    # Global statistics
    global_mean: float
    global_std: float
    global_max: float
    morans_i: float
    
    # GNN embeddings
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    
    # Region information
    cluster_labels: np.ndarray
    region_stats: Dict[int, RegionStats]
    n_regions: int
    
    # Raw data
    coordinates: np.ndarray
    aligned_coordinates: Optional[np.ndarray]
    raw_values: np.ndarray


# =============================================================================
# GNN MODEL
# =============================================================================

class SpatialGNN(nn.Module):
    """Spatial GNN for embeddings"""
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 3,
        projection_dim: int = 64
    ):
        super().__init__()
        
        self.projection_dim = projection_dim
        self.input_projections = nn.ModuleDict()
        
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Sequential(
                nn.Linear(input_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
        
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        self.convs.append(GATConv(projection_dim, hidden_dim // num_heads,
                                  heads=num_heads, concat=True, dropout=0.1))
        self.norms.append(nn.LayerNorm(hidden_dim))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim, hidden_dim // num_heads,
                                      heads=num_heads, concat=True, dropout=0.1))
            self.norms.append(nn.LayerNorm(hidden_dim))
        
        self.convs.append(GATConv(hidden_dim, embedding_dim, heads=1, concat=False, dropout=0.1))
        self.norms.append(nn.LayerNorm(embedding_dim))
        
        self.importance_head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        self.pool_attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, projection_dim)
        )
    
    def _get_projection(self, dim):
        key = str(dim)
        if key not in self.input_projections:
            device = next(self.parameters()).device
            self.input_projections[key] = nn.Sequential(
                nn.Linear(dim, self.projection_dim),
                nn.LayerNorm(self.projection_dim),
                nn.GELU()
            ).to(device)
        return self.input_projections[key]
    
    def forward(self, x, edge_index, bio_importance=None):
        h_input = self._get_projection(x.size(1))(x)
        
        h = h_input
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index)
            h = norm(h)
            h = F.gelu(h)
        
        node_embeddings = h
        
        learned_imp = self.importance_head(node_embeddings).squeeze(-1)
        if bio_importance is not None:
            node_importance = 0.5 * learned_imp + 0.5 * bio_importance
        else:
            node_importance = learned_imp
        node_importance = node_importance / (node_importance.max() + 1e-8)
        
        weight = node_importance / (node_importance.sum() + 1e-8) * x.size(0)
        weighted_emb = node_embeddings * weight.unsqueeze(-1)
        
        attn = F.softmax(self.pool_attention(node_embeddings).squeeze(-1), dim=0)
        global_emb = (attn.unsqueeze(-1) * weighted_emb).sum(dim=0)
        
        reconstructed = self.decoder(node_embeddings)
        
        return {
            'node_embeddings': node_embeddings,
            'global_embedding': global_emb,
            'node_importance': node_importance,
            'reconstructed': reconstructed,
            'h_input': h_input
        }


class SpatialLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, output, edge_index):
        recon = F.mse_loss(output['reconstructed'], output['h_input'])
        src, dst = edge_index[0], edge_index[1]
        smooth = ((output['node_embeddings'][src] - output['node_embeddings'][dst]) ** 2).mean()
        return recon + 0.3 * smooth


# =============================================================================
# REGION-SPECIFIC SIMILARITY
# =============================================================================

def compute_region_similarity(
    gene_region: RegionStats,
    mz_region: RegionStats
) -> dict:
    """
    Compute similarity between corresponding regions using FULL region data.
    """
    if gene_region is None or mz_region is None:
        return {'region_score': 0, 'valid': False}
    
    # === VALUE STATISTICS SIMILARITY ===
    # Mean similarity (normalized)
    max_mean = max(abs(gene_region.mean_value), abs(mz_region.mean_value), 1e-8)
    mean_sim = 1 - abs(gene_region.mean_value - mz_region.mean_value) / max_mean
    mean_sim = max(0, mean_sim)
    
    # Std similarity
    max_std = max(gene_region.std_value, mz_region.std_value, 1e-8)
    std_sim = 1 - abs(gene_region.std_value - mz_region.std_value) / max_std
    std_sim = max(0, std_sim)
    
    # Median similarity
    max_median = max(abs(gene_region.median_value), abs(mz_region.median_value), 1e-8)
    median_sim = 1 - abs(gene_region.median_value - mz_region.median_value) / max_median
    median_sim = max(0, median_sim)
    
    # === DISTRIBUTION SIMILARITY ===
    # Histogram correlation
    if gene_region.value_histogram.std() > 0 and mz_region.value_histogram.std() > 0:
        hist_corr, _ = pearsonr(gene_region.value_histogram, mz_region.value_histogram)
        hist_corr = hist_corr if not np.isnan(hist_corr) else 0
    else:
        hist_corr = 0
    
    # Percentile similarity (compare distribution shapes)
    if gene_region.percentiles.std() > 0 and mz_region.percentiles.std() > 0:
        # Normalize percentiles to [0,1] range
        g_pct = (gene_region.percentiles - gene_region.percentiles.min()) / \
                (gene_region.percentiles.max() - gene_region.percentiles.min() + 1e-8)
        m_pct = (mz_region.percentiles - mz_region.percentiles.min()) / \
                (mz_region.percentiles.max() - mz_region.percentiles.min() + 1e-8)
        pct_corr, _ = pearsonr(g_pct, m_pct)
        pct_corr = pct_corr if not np.isnan(pct_corr) else 0
    else:
        pct_corr = 0
    
    # === SPATIAL DISTRIBUTION SIMILARITY ===
    # Value correlation within region (using gridded comparison)
    if gene_region.coords is not None and mz_region.coords is not None:
        value_corr = compute_region_value_correlation(gene_region, mz_region)
    else:
        value_corr = 0
    
    # Centroid similarity (how close are centroids relative to region size)
    gene_size = max(gene_region.x_range[1] - gene_region.x_range[0],
                    gene_region.y_range[1] - gene_region.y_range[0], 1e-8)
    mz_size = max(mz_region.x_range[1] - mz_region.x_range[0],
                  mz_region.y_range[1] - mz_region.y_range[0], 1e-8)
    avg_size = (gene_size + mz_size) / 2
    
    centroid_dist = np.linalg.norm(gene_region.centroid - mz_region.centroid)
    centroid_sim = 1 / (1 + centroid_dist / avg_size)
    
    # Value-weighted centroid similarity
    vw_centroid_dist = np.linalg.norm(gene_region.value_weighted_centroid - mz_region.value_weighted_centroid)
    vw_centroid_sim = 1 / (1 + vw_centroid_dist / avg_size)
    
    # Spatial spread similarity
    max_spread = max(gene_region.spatial_spread, mz_region.spatial_spread, 1e-8)
    spread_sim = 1 - abs(gene_region.spatial_spread - mz_region.spatial_spread) / max_spread
    spread_sim = max(0, spread_sim)
    
    # === COMBINED REGION SCORE ===
    region_score = (
        0.25 * max(value_corr, 0) +         # Spatial value correlation (main)
        0.20 * max(hist_corr, 0) +           # Distribution similarity
        0.15 * vw_centroid_sim +             # Value-weighted centroid match
        0.10 * centroid_sim +                # Centroid proximity
        0.10 * mean_sim +                    # Mean value similarity
        0.10 * median_sim +                  # Median similarity
        0.05 * std_sim +                     # Std similarity
        0.05 * max(pct_corr, 0)              # Percentile shape similarity
    )
    
    return {
        'region_score': region_score,
        'value_corr': value_corr,
        'hist_corr': hist_corr,
        'centroid_sim': centroid_sim,
        'vw_centroid_sim': vw_centroid_sim,
        'mean_sim': mean_sim,
        'median_sim': median_sim,
        'std_sim': std_sim,
        'spread_sim': spread_sim,
        'pct_corr': pct_corr,
        'gene_n_points': gene_region.n_points,
        'mz_n_points': mz_region.n_points,
        'gene_mean': gene_region.mean_value,
        'mz_mean': mz_region.mean_value,
        'valid': True
    }


def compute_region_value_correlation(
    gene_region: RegionStats,
    mz_region: RegionStats,
    grid_res: int = 20
) -> float:
    """Compute value correlation within a region using grid interpolation"""
    # Common bounding box
    x_min = min(gene_region.x_range[0], mz_region.x_range[0])
    x_max = max(gene_region.x_range[1], mz_region.x_range[1])
    y_min = min(gene_region.y_range[0], mz_region.y_range[0])
    y_max = max(gene_region.y_range[1], mz_region.y_range[1])
    
    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    
    gene_grid = griddata(gene_region.coords, gene_region.values, (grid_x, grid_y), method='linear')
    mz_grid = griddata(mz_region.coords, mz_region.values, (grid_x, grid_y), method='linear')
    
    mask = ~(np.isnan(gene_grid) | np.isnan(mz_grid))
    
    if mask.sum() < 10:
        return 0
    
    corr, _ = pearsonr(gene_grid[mask], mz_grid[mask])
    return corr if not np.isnan(corr) else 0


def compute_overall_similarity(
    gene_sig: SpatialSignature,
    mz_sig: SpatialSignature,
    region_mapping: Dict[int, int]
) -> dict:
    """
    Compute overall similarity using region-specific comparisons.
    
    Combines:
    1. Region-specific similarities (weighted by region size)
    2. Global embedding similarity
    3. Overall pattern correlation
    """
    # Region-specific similarities
    region_scores = []
    region_weights = []
    region_details = {}
    
    for gene_cluster, mz_cluster in region_mapping.items():
        if gene_cluster in gene_sig.region_stats and mz_cluster in mz_sig.region_stats:
            gene_region = gene_sig.region_stats[gene_cluster]
            mz_region = mz_sig.region_stats[mz_cluster]
            
            sim = compute_region_similarity(gene_region, mz_region)
            
            if sim['valid']:
                region_scores.append(sim['region_score'])
                # Weight by geometric mean of region sizes
                weight = np.sqrt(gene_region.n_points * mz_region.n_points)
                region_weights.append(weight)
                region_details[f'region_{gene_cluster}_to_{mz_cluster}'] = sim
    
    # Weighted average of region scores
    if len(region_scores) > 0:
        region_weights = np.array(region_weights)
        region_weights = region_weights / region_weights.sum()
        avg_region_score = np.sum(np.array(region_scores) * region_weights)
    else:
        avg_region_score = 0
    
    # Global embedding similarity
    g_emb = gene_sig.global_embedding.flatten()
    m_emb = mz_sig.global_embedding.flatten()
    emb_cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
    
    # Moran's I similarity
    morans_diff = abs(gene_sig.morans_i - mz_sig.morans_i)
    morans_sim = 1 / (1 + morans_diff)
    
    # Combined score
    combined_score = (
        0.55 * avg_region_score +      # Region-specific (main)
        0.25 * emb_cosine +             # Global embedding
        0.10 * morans_sim +             # Spatial autocorrelation
        0.10 * (len(region_scores) / max(gene_sig.n_regions, mz_sig.n_regions))  # Coverage
    )
    
    return {
        'combined_score': combined_score,
        'avg_region_score': avg_region_score,
        'embedding_cosine': emb_cosine,
        'morans_similarity': morans_sim,
        'n_matched_regions': len(region_scores),
        'region_details': region_details
    }


# =============================================================================
# MAIN MATCHER
# =============================================================================

class RegionMatcher:
    """Gene-to-m/z matcher using Leiden clustering for region-specific comparison"""
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v8',
        n_clusters: int = N_CLUSTERS,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        self.n_clusters = n_clusters
        
        subdirs = ['saliency_maps', 'gene_visualizations', 'match_visualizations',
                   'embeddings', 'training_curves', 'region_maps']
        for subdir in subdirs:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.rna_clusters = {}  # sample_id -> cluster labels
        self.msi_clusters = {}
        self.rna_model = None
        self.msi_model = None
    
    def load_all_data(self):
        """Load all data"""
        print("Loading RNA-seq data...")
        print(f"  Pixel size: {RNA_PIXEL_SIZE} μm (hexagonal)")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print(f"\nLoading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def compute_all_clusters(self):
        """Compute Leiden clusters for all samples - exactly 3 clusters each"""
        print(f"\nComputing Leiden clusters (target: {N_CLUSTERS} clusters per sample)...")
        
        print("\nRNA clusters:")
        for sample_id, adata in self.rna_data.items():
            try:
                clusters = compute_spatial_leiden_fixed_clusters(adata, n_clusters=N_CLUSTERS)
                self.rna_clusters[sample_id] = clusters
                n_clusters = len(np.unique(clusters))
                print(f"  {sample_id}: {n_clusters} clusters")
            except Exception as e:
                print(f"  {sample_id}: Leiden failed ({e}), using spatial-only")
                try:
                    # Fallback: spatial-only clustering
                    coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                    clusters = compute_spatial_only_leiden_fixed(coords, n_clusters=N_CLUSTERS)
                    self.rna_clusters[sample_id] = clusters
                    print(f"  {sample_id}: {len(np.unique(clusters))} clusters (spatial-only)")
                except Exception as e2:
                    print(f"  {sample_id}: Spatial-only also failed ({e2}), using simple k-means")
                    # Final fallback: simple k-means
                    from sklearn.cluster import KMeans
                    coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
                    clusters = kmeans.fit_predict(coords)
                    self.rna_clusters[sample_id] = clusters
                    print(f"  {sample_id}: {len(np.unique(clusters))} clusters (k-means fallback)")
        
        print("\nMSI clusters:")
        for sample_id, adata in self.msi_data.items():
            try:
                clusters = compute_spatial_leiden_fixed_clusters(adata, n_clusters=N_CLUSTERS)
                self.msi_clusters[sample_id] = clusters
                n_clusters = len(np.unique(clusters))
                print(f"  {sample_id}: {n_clusters} clusters")
            except Exception as e:
                print(f"  {sample_id}: Leiden failed ({e}), using spatial-only")
                try:
                    coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                    clusters = compute_spatial_only_leiden_fixed(coords, n_clusters=N_CLUSTERS)
                    self.msi_clusters[sample_id] = clusters
                    print(f"  {sample_id}: {len(np.unique(clusters))} clusters (spatial-only)")
                except Exception as e2:
                    print(f"  {sample_id}: Spatial-only also failed ({e2}), using simple k-means")
                    from sklearn.cluster import KMeans
                    coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
                    clusters = kmeans.fit_predict(coords)
                    self.msi_clusters[sample_id] = clusters
                    print(f"  {sample_id}: {len(np.unique(clusters))} clusters (k-means fallback)")
    
    def visualize_clusters(self, sample_id: str, modality: str):
        """Visualize clusters for a sample"""
        if modality == 'rna':
            adata = self.rna_data[sample_id]
            clusters = self.rna_clusters[sample_id]
        else:
            adata = self.msi_data[sample_id]
            clusters = self.msi_clusters[sample_id]
        
        coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
        
        fig, ax = plt.subplots(figsize=(10, 8))
        
        n_clusters = len(np.unique(clusters))
        cmap = plt.cm.get_cmap('tab10')
        
        scatter = ax.scatter(coords[:, 0], coords[:, 1], c=clusters, 
                            cmap=cmap, s=5 if modality == 'msi' else 15, alpha=0.7)
        
        # Add centroid labels
        for c in np.unique(clusters):
            mask = clusters == c
            centroid = coords[mask].mean(axis=0)
            n_points = mask.sum()
            ax.annotate(f'R{c}\n({n_points})', centroid, fontsize=12, fontweight='bold',
                       ha='center', va='center',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        
        ax.set_title(f'{modality.upper()} {sample_id} - {n_clusters} Clusters (fixed)',
                    fontweight='bold')
        plt.colorbar(scatter, ax=ax, label='Cluster')
        
        save_path = os.path.join(self.output_dir, 'region_maps', f'{modality}_{sample_id}_clusters.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    
    def build_graph(self, coords, n_neighbors):
        """Build spatial graph"""
        nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
        nn.fit(coords)
        distances, indices = nn.kneighbors(coords)
        
        median_dist = np.median(distances[:, 1])
        max_dist = median_dist * 1.5
        
        edges = set()
        for i in range(len(coords)):
            for j_idx in range(1, n_neighbors + 1):
                if distances[i, j_idx] <= max_dist:
                    j = indices[i, j_idx]
                    edges.add((i, j))
                    edges.add((j, i))
        
        return torch.tensor(list(edges), dtype=torch.long).t().contiguous()
    
    def compute_bio_importance(self, coords, values, k):
        """Biological importance"""
        nn = NearestNeighbors(n_neighbors=k + 1)
        nn.fit(coords)
        _, indices = nn.kneighbors(coords)
        
        local_var = np.array([np.var(values[indices[i, 1:]]) for i in range(len(coords))])
        local_var = (local_var - local_var.min()) / (local_var.max() - local_var.min() + 1e-8)
        
        val_norm = (values - values.min()) / (values.max() - values.min() + 1e-8)
        
        return 0.5 * local_var + 0.5 * val_norm
    
    def compute_morans_i(self, coords, values, k=8):
        """Moran's I"""
        norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
        
        nn = NearestNeighbors(n_neighbors=k + 1)
        nn.fit(norm)
        _, indices = nn.kneighbors(norm)
        
        n = len(values)
        mean_val = values.mean()
        denom = np.sum((values - mean_val) ** 2)
        
        if denom == 0:
            return 0
        
        numer = 0
        w_sum = 0
        
        for i in range(n):
            for j in indices[i, 1:]:
                numer += (values[i] - mean_val) * (values[j] - mean_val)
                w_sum += 1
        
        return (n / w_sum) * (numer / denom) if w_sum > 0 else 0
    
    def prepare_data(self, coords, features, n_neighbors):
        """Prepare graph data"""
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        bio_imp = np.zeros(len(coords))
        for col in range(features.shape[1]):
            bio_imp += self.compute_bio_importance(coords, features[:, col], n_neighbors)
        bio_imp /= features.shape[1]
        
        scaler = RobustScaler()
        features_scaled = scaler.fit_transform(features)
        
        edge_index = self.build_graph(coords, n_neighbors)
        
        return Data(x=torch.tensor(features_scaled, dtype=torch.float32), edge_index=edge_index), bio_imp
    
    def train_model(self, data_dict, bio_dict, model_type, epochs=150):
        """Train model"""
        print(f"\nTraining {model_type.upper()} model...")
        
        first_data = list(data_dict.values())[0]
        model = SpatialGNN(input_dim=first_data.x.size(1)).to(self.device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        loss_fn = SpatialLoss()
        
        losses = []
        model.train()
        
        for epoch in range(epochs):
            total_loss = 0
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                bio_imp = torch.tensor(bio_dict[sample_id], dtype=torch.float32, device=self.device)
                
                optimizer.zero_grad()
                output = model(data.x, data.edge_index, bio_imp)
                loss = loss_fn(output, data.edge_index)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            losses.append(total_loss / len(data_dict))
            if (epoch + 1) % 30 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {losses[-1]:.4f}")
        
        print(f"  Final loss: {losses[-1]:.4f}")
        
        plt.figure(figsize=(10, 5))
        plt.plot(losses)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_type.upper()} Training')
        plt.savefig(os.path.join(self.output_dir, 'training_curves', f'{model_type}_loss.png'), dpi=150)
        plt.close()
        
        return model
    
    def extract_signature(
        self,
        coords: np.ndarray,
        values: np.ndarray,
        sample_id: str,
        feature_name: str,
        feature_type: str,
        model: SpatialGNN,
        n_neighbors: int,
        cluster_labels: np.ndarray,
        msi_coords: Optional[np.ndarray] = None
    ) -> SpatialSignature:
        """Extract signature with region information"""
        
        # GNN embeddings
        bio_imp = self.compute_bio_importance(coords, values, n_neighbors)
        data, _ = self.prepare_data(coords, values, n_neighbors)
        data = data.to(self.device)
        bio_tensor = torch.tensor(bio_imp, dtype=torch.float32, device=self.device)
        
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, bio_tensor)
        
        node_importance = output['node_importance'].cpu().numpy()
        
        # Aligned coordinates for RNA
        aligned = None
        if feature_type == 'gene' and msi_coords is not None:
            aligned = align_rna_to_msi(coords, msi_coords)
        
        # Use aligned coordinates for region stats if available
        region_coords = aligned if aligned is not None else coords
        
        # Compute region statistics
        region_stats = compute_region_stats(
            region_coords, values, node_importance, cluster_labels
        )
        
        # Global statistics
        morans = self.compute_morans_i(coords, values, n_neighbors)
        
        return SpatialSignature(
            sample_id=sample_id,
            feature_name=feature_name,
            feature_type=feature_type,
            global_mean=values.mean(),
            global_std=values.std(),
            global_max=values.max(),
            morans_i=morans,
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['node_embeddings'].cpu().numpy(),
            node_importance=node_importance,
            cluster_labels=cluster_labels,
            region_stats=region_stats,
            n_regions=len(region_stats),
            coordinates=coords,
            aligned_coordinates=aligned,
            raw_values=values
        )
    
    def visualize_signature(self, sig: SpatialSignature, save_path: str):
        """Visualize signature with full region statistics"""
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        
        # Row 0: Values and importance
        im1 = axes[0, 0].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.raw_values, cmap='viridis', s=10)
        axes[0, 0].set_title(f'{sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        im2 = axes[0, 1].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.node_importance, cmap='hot', s=10)
        axes[0, 1].set_title('Importance', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # Clusters with region labels
        im3 = axes[0, 2].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.cluster_labels, cmap='tab10', s=10)
        # Add centroid labels
        for cluster_id, stats in sig.region_stats.items():
            axes[0, 2].annotate(f'R{cluster_id}', stats.centroid, fontsize=10, fontweight='bold',
                               ha='center', va='center',
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        axes[0, 2].set_title(f'Leiden Clusters ({sig.n_regions} regions)', fontweight='bold')
        plt.colorbar(im3, ax=axes[0, 2])
        
        # Aligned if available
        if sig.aligned_coordinates is not None:
            im4 = axes[0, 3].scatter(sig.aligned_coordinates[:, 0], sig.aligned_coordinates[:, 1],
                                      c=sig.raw_values, cmap='viridis', s=10)
            axes[0, 3].set_title('Aligned (180° rotated)', fontweight='bold')
            plt.colorbar(im4, ax=axes[0, 3])
        else:
            axes[0, 3].axis('off')
        
        # Row 1: Region statistics
        # Region means
        region_means = []
        region_labels = []
        region_stds = []
        for cluster_id, stats in sorted(sig.region_stats.items()):
            region_means.append(stats.mean_value)
            region_stds.append(stats.std_value)
            region_labels.append(f'R{cluster_id}\n({stats.n_points})')
        
        if len(region_means) > 0:
            x = np.arange(len(region_means))
            axes[1, 0].bar(x, region_means, yerr=region_stds, capsize=3, alpha=0.7)
            axes[1, 0].set_xticks(x)
            axes[1, 0].set_xticklabels(region_labels, fontsize=8)
            axes[1, 0].set_title('Mean ± Std by Region', fontweight='bold')
        
        # Region medians with quartile range
        region_medians = []
        region_q25 = []
        region_q75 = []
        for cluster_id, stats in sorted(sig.region_stats.items()):
            region_medians.append(stats.median_value)
            region_q25.append(stats.percentiles[1])  # 25th percentile
            region_q75.append(stats.percentiles[3])  # 75th percentile
        
        if len(region_medians) > 0:
            x = np.arange(len(region_medians))
            axes[1, 1].bar(x, region_medians, alpha=0.7, color='green')
            axes[1, 1].errorbar(x, region_medians, 
                               yerr=[np.array(region_medians) - np.array(region_q25),
                                    np.array(region_q75) - np.array(region_medians)],
                               fmt='none', color='black', capsize=3)
            axes[1, 1].set_xticks(x)
            axes[1, 1].set_xticklabels([f'R{c}' for c in sorted(sig.region_stats.keys())], fontsize=8)
            axes[1, 1].set_title('Median (IQR) by Region', fontweight='bold')
        
        # Region total values (contribution)
        region_totals = []
        for cluster_id, stats in sorted(sig.region_stats.items()):
            region_totals.append(stats.total_value)
        
        if len(region_totals) > 0:
            total_sum = sum(region_totals)
            percentages = [t / total_sum * 100 for t in region_totals]
            colors = plt.cm.tab10(np.linspace(0, 1, len(region_totals)))
            axes[1, 2].pie(percentages, labels=[f'R{c}' for c in sorted(sig.region_stats.keys())],
                          autopct='%1.1f%%', colors=colors)
            axes[1, 2].set_title('Total Value Contribution', fontweight='bold')
        
        # Statistics text
        stats_text = f"""
Sample: {sig.sample_id}
Feature: {sig.feature_name}
Type: {sig.feature_type}

Global Stats:
  Mean: {sig.global_mean:.4f}
  Std: {sig.global_std:.4f}
  Max: {sig.global_max:.4f}
  Moran's I: {sig.morans_i:.4f}

Regions: {sig.n_regions}
Total points: {len(sig.raw_values)}
        """
        axes[1, 3].text(0.1, 0.9, stats_text, transform=axes[1, 3].transAxes,
                       fontsize=10, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[1, 3].axis('off')
        
        plt.suptitle(f'{sig.feature_type.upper()}: {sig.feature_name} ({sig.sample_id})',
                    fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_match(self, gene_sig, mz_sig, similarity, region_mapping, save_path):
        """Visualize match with full region comparison"""
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        
        # Row 0: Gene
        im1 = axes[0, 0].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name}', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        im2 = axes[0, 1].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.cluster_labels, cmap='tab10', s=15)
        for cluster_id, stats in gene_sig.region_stats.items():
            axes[0, 1].annotate(f'R{cluster_id}', stats.centroid, fontsize=9, fontweight='bold',
                               ha='center', va='center',
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        axes[0, 1].set_title(f'Gene Clusters ({gene_sig.n_regions})', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        if gene_sig.aligned_coordinates is not None:
            im3 = axes[0, 2].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                                      c=gene_sig.raw_values, cmap='viridis', s=15)
            axes[0, 2].set_title('Gene Aligned', fontweight='bold')
            plt.colorbar(im3, ax=axes[0, 2])
        
        # Gene value-weighted centroids
        if gene_sig.aligned_coordinates is not None:
            axes[0, 3].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                              c='lightgray', s=5, alpha=0.3)
            for cluster_id, stats in gene_sig.region_stats.items():
                axes[0, 3].scatter(*stats.value_weighted_centroid, c='red', s=100, marker='*',
                                  edgecolors='black', linewidths=1)
                axes[0, 3].annotate(f'R{cluster_id}', stats.value_weighted_centroid, fontsize=8)
            axes[0, 3].set_title('Gene Value-Weighted Centroids', fontweight='bold')
        
        # Row 1: m/z
        im4 = axes[1, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.raw_values, cmap='viridis', s=5)
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 0])
        
        im5 = axes[1, 1].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.cluster_labels, cmap='tab10', s=5)
        for cluster_id, stats in mz_sig.region_stats.items():
            axes[1, 1].annotate(f'R{cluster_id}', stats.centroid, fontsize=9, fontweight='bold',
                               ha='center', va='center',
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        axes[1, 1].set_title(f'm/z Clusters ({mz_sig.n_regions})', fontweight='bold')
        plt.colorbar(im5, ax=axes[1, 1])
        
        # m/z value-weighted centroids
        axes[1, 2].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                          c='lightgray', s=3, alpha=0.3)
        for cluster_id, stats in mz_sig.region_stats.items():
            axes[1, 2].scatter(*stats.value_weighted_centroid, c='blue', s=100, marker='*',
                              edgecolors='black', linewidths=1)
            axes[1, 2].annotate(f'R{cluster_id}', stats.value_weighted_centroid, fontsize=8)
        axes[1, 2].set_title('m/z Value-Weighted Centroids', fontweight='bold')
        
        # Combined overlay with region matching
        if gene_sig.aligned_coordinates is not None:
            axes[1, 3].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                              c='lightblue', s=2, alpha=0.3, label='m/z')
            axes[1, 3].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                              c='lightcoral', s=5, alpha=0.3, label='Gene')
            # Draw lines between matched region centroids
            for gene_c, mz_c in region_mapping.items():
                if gene_c in gene_sig.region_stats and mz_c in mz_sig.region_stats:
                    g_cent = gene_sig.region_stats[gene_c].centroid
                    m_cent = mz_sig.region_stats[mz_c].centroid
                    axes[1, 3].plot([g_cent[0], m_cent[0]], [g_cent[1], m_cent[1]],
                                   'g-', linewidth=2, alpha=0.7)
                    axes[1, 3].scatter(*g_cent, c='red', s=80, marker='o', zorder=5)
                    axes[1, 3].scatter(*m_cent, c='blue', s=80, marker='s', zorder=5)
            axes[1, 3].set_title('Region Mapping', fontweight='bold')
            axes[1, 3].legend()
        
        # Row 2: Region comparison and metrics
        # Region scores
        region_scores = []
        region_labels = []
        for gene_c, mz_c in region_mapping.items():
            key = f'region_{gene_c}_to_{mz_c}'
            if key in similarity.get('region_details', {}):
                region_scores.append(similarity['region_details'][key]['region_score'])
                region_labels.append(f'G{gene_c}→M{mz_c}')
        
        if len(region_scores) > 0:
            colors = plt.cm.RdYlGn(np.array(region_scores))
            axes[2, 0].bar(range(len(region_scores)), region_scores, color=colors)
            axes[2, 0].set_xticks(range(len(region_scores)))
            axes[2, 0].set_xticklabels(region_labels, rotation=45, ha='right')
            axes[2, 0].set_ylim(0, 1)
            axes[2, 0].set_title('Region Match Scores', fontweight='bold')
            axes[2, 0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
        
        # Region mean comparison (full region)
        gene_means = []
        mz_means = []
        labels = []
        for gene_c, mz_c in region_mapping.items():
            if gene_c in gene_sig.region_stats and mz_c in mz_sig.region_stats:
                gene_means.append(gene_sig.region_stats[gene_c].mean_value)
                mz_means.append(mz_sig.region_stats[mz_c].mean_value)
                labels.append(f'G{gene_c}/M{mz_c}')
        
        if len(gene_means) > 0:
            x = np.arange(len(gene_means))
            width = 0.35
            axes[2, 1].bar(x - width/2, gene_means, width, label='Gene', alpha=0.7, color='red')
            axes[2, 1].bar(x + width/2, mz_means, width, label='m/z', alpha=0.7, color='blue')
            axes[2, 1].set_xticks(x)
            axes[2, 1].set_xticklabels(labels, fontsize=8)
            axes[2, 1].set_title('Region Mean Values', fontweight='bold')
            axes[2, 1].legend()
        
        # Region value correlation scores
        value_corrs = []
        for gene_c, mz_c in region_mapping.items():
            key = f'region_{gene_c}_to_{mz_c}'
            if key in similarity.get('region_details', {}):
                value_corrs.append(similarity['region_details'][key].get('value_corr', 0))
        
        if len(value_corrs) > 0:
            colors = ['green' if v > 0 else 'red' for v in value_corrs]
            axes[2, 2].bar(range(len(value_corrs)), value_corrs, color=colors, alpha=0.7)
            axes[2, 2].set_xticks(range(len(value_corrs)))
            axes[2, 2].set_xticklabels(region_labels, rotation=45, ha='right', fontsize=8)
            axes[2, 2].set_ylim(-1, 1)
            axes[2, 2].axhline(y=0, color='gray', linestyle='-', alpha=0.5)
            axes[2, 2].set_title('Region Value Correlations', fontweight='bold')
        
        # Metrics
        metrics_text = f"""
COMBINED SCORE: {similarity['combined_score']:.4f}
═══════════════════════════════════

REGION-BASED (55%):
  Avg region score:     {similarity['avg_region_score']:.4f}
  Matched regions:      {similarity['n_matched_regions']}

GLOBAL (45%):
  Embedding cosine:     {similarity['embedding_cosine']:.4f}
  Moran's I sim:        {similarity['morans_similarity']:.4f}

SPATIAL STATS:
  Gene Moran's I:       {gene_sig.morans_i:.4f}
  m/z Moran's I:        {mz_sig.morans_i:.4f}
  Gene regions:         {gene_sig.n_regions}
  m/z regions:          {mz_sig.n_regions}

Region Mapping:
{chr(10).join([f'  G{g} → M{m}' for g, m in region_mapping.items()])}
        """
        axes[2, 3].text(0.05, 0.95, metrics_text, transform=axes[2, 3].transAxes,
                       fontsize=9, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[2, 3].axis('off')
        
        plt.suptitle(f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name} | Score: {similarity["combined_score"]:.3f}',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def find_matches(self, gene_sig, mz_sigs, region_mapping, top_k=20):
        """Find matches using region-specific comparison"""
        matches = []
        
        for mz_name, mz_sig in mz_sigs.items():
            sim = compute_overall_similarity(gene_sig, mz_sig, region_mapping)
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                'combined_score': sim['combined_score'],
                'avg_region_score': sim['avg_region_score'],
                'embedding_cosine': sim['embedding_cosine'],
                'morans_similarity': sim['morans_similarity'],
                'n_matched_regions': sim['n_matched_regions']
            })
        
        return pd.DataFrame(matches).sort_values('combined_score', ascending=False).head(top_k)
    
    def run_analysis(self, epochs=150, top_k=20):
        """Run analysis"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V8")
        print("Region-Specific Comparison with Leiden Clustering")
        print(f"Fixed clusters per sample: {N_CLUSTERS}")
        print("="*70)
        
        # Compute clusters
        self.compute_all_clusters()
        
        # Visualize clusters
        print("\nVisualizing clusters...")
        for sample_id in list(self.rna_data.keys())[:4]:
            self.visualize_clusters(sample_id, 'rna')
        for sample_id in list(self.msi_data.keys())[:4]:
            self.visualize_clusters(sample_id, 'msi')
        
        # Gene availability
        gene_avail = {}
        for gene in AAD_TARGET_GENES:
            gene_avail[gene] = {s: gene in self.rna_data[s].var_names
                               for s in RNA_SAMPLE_IDS if s in self.rna_data}
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            n = sum(gene_avail[gene].values())
            print(f"  {gene}: {n}/{len(RNA_SAMPLE_IDS)}")
        
        # Train
        print("\n" + "-"*70)
        print("PHASE 1: Training")
        print("-"*70)
        
        # RNA
        print(f"\nRNA (6 neighbors)...")
        rna_train, rna_bio = {}, {}
        for sid in self.rna_data.keys():
            try:
                adata = self.rna_data[sid]
                coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                features = adata.X[:, :min(50, adata.X.shape[1])]
                if hasattr(features, 'toarray'):
                    features = features.toarray()
                data, bio = self.prepare_data(coords, features, 6)
                rna_train[sid] = data
                rna_bio[sid] = bio
                print(f"  {sid}: {data.x.shape}")
                if len(rna_train) >= 4:  # Use first 4 for training
                    break
            except Exception as e:
                print(f"  {sid}: Failed - {e}")
        
        if len(rna_train) == 0:
            raise ValueError("No RNA data available for training!")
        
        self.rna_model = self.train_model(rna_train, rna_bio, 'rna', epochs)
        
        # MSI
        print(f"\nMSI (8 neighbors)...")
        msi_train, msi_bio = {}, {}
        for sid in self.msi_data.keys():
            try:
                adata = self.msi_data[sid]
                coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                features = adata.X[:, :min(50, adata.X.shape[1])]
                if hasattr(features, 'toarray'):
                    features = features.toarray()
                data, bio = self.prepare_data(coords, features, 8)
                msi_train[sid] = data
                msi_bio[sid] = bio
                print(f"  {sid}: {data.x.shape}")
                if len(msi_train) >= 4:  # Use first 4 for training
                    break
            except Exception as e:
                print(f"  {sid}: Failed - {e}")
        
        if len(msi_train) == 0:
            raise ValueError("No MSI data available for training!")
        
        self.msi_model = self.train_model(msi_train, msi_bio, 'msi', epochs)
        
        # Match
        print("\n" + "-"*70)
        print("PHASE 2: Region-Specific Matching")
        print("-"*70)
        
        all_results = []
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_avail[gene].items() if a]
            if not available:
                continue
            
            for rna_sample in available:
                print(f"\n  {rna_sample}")
                
                if rna_sample not in self.rna_clusters or rna_sample not in self.msi_data:
                    print("    Skipping - missing clusters or MSI data")
                    continue
                
                # Get data
                adata = self.rna_data[rna_sample]
                rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                gene_idx = list(adata.var_names).index(gene)
                gene_expr = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, 'toarray') \
                           else adata.X[:, gene_idx].flatten()
                
                msi_adata = self.msi_data[rna_sample]
                msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
                
                # Get aligned gene coordinates for region matching
                gene_aligned = align_rna_to_msi(rna_coords, msi_coords)
                
                # Match regions
                region_mapping = match_regions_by_centroid(
                    gene_aligned, self.rna_clusters[rna_sample],
                    msi_coords, self.msi_clusters[rna_sample]
                )
                print(f"    Region mapping: {region_mapping}")
                
                # Extract gene signature
                gene_sig = self.extract_signature(
                    rna_coords, gene_expr, rna_sample, gene, 'gene',
                    self.rna_model, 6, self.rna_clusters[rna_sample], msi_coords
                )
                
                # Save visualizations
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'saliency_maps', f'{gene}_{rna_sample}.png')
                )
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'gene_visualizations', f'{gene}_{rna_sample}.png')
                )
                
                # Save embeddings
                with open(os.path.join(self.output_dir, 'embeddings', f'{gene}_{rna_sample}.pkl'), 'wb') as f:
                    pickle.dump({
                        'global_embedding': gene_sig.global_embedding,
                        'node_importance': gene_sig.node_importance,
                        'cluster_labels': gene_sig.cluster_labels,
                        'region_stats': {k: {
                            'mean': v.mean_value,
                            'median': v.median_value,
                            'std': v.std_value,
                            'max': v.max_value,
                            'n_points': v.n_points,
                            'centroid': v.centroid.tolist(),
                            'value_weighted_centroid': v.value_weighted_centroid.tolist()
                        } for k, v in gene_sig.region_stats.items()}
                    }, f)
                
                # Extract m/z signatures
                print(f"    Matching vs MSI...")
                msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
                mz_names = list(msi_adata.var_names)
                
                mz_sigs = {}
                for i, mz_name in enumerate(mz_names):
                    mz_sigs[mz_name] = self.extract_signature(
                        msi_coords, msi_data[:, i], rna_sample, mz_name, 'mz',
                        self.msi_model, 8, self.msi_clusters[rna_sample]
                    )
                    if (i + 1) % 100 == 0:
                        print(f"      {i+1}/{len(mz_names)}")
                
                # Find matches
                matches = self.find_matches(gene_sig, mz_sigs, region_mapping, top_k)
                all_results.append(matches)
                
                if len(matches) > 0:
                    print(f"\n    Top 5:")
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        print(f"      {idx+1}. m/z {m['mz_feature']}: {m['combined_score']:.3f} "
                              f"(region={m['avg_region_score']:.3f})")
                    
                    # Visualize top match
                    top = matches.iloc[0]
                    top_mz = mz_sigs[top['mz_feature']]
                    sim = compute_overall_similarity(gene_sig, top_mz, region_mapping)
                    
                    self.visualize_match(
                        gene_sig, top_mz, sim, region_mapping,
                        os.path.join(self.output_dir, 'match_visualizations', f'{gene}_{rna_sample}_top.png')
                    )
        
        # Save results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            results.to_csv(os.path.join(self.output_dir, 'gene_to_mz_matches_v8.csv'), index=False)
            
            print("\n" + "="*70)
            print("TOP MATCHES")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gr = results[results['gene'] == gene]
                if len(gr) > 0:
                    t = gr.iloc[0]
                    print(f"\n{gene}: m/z {t['mz_feature']} ({t['rna_sample']}) = {t['combined_score']:.3f}")
            
            return results
        
        return None


def main():
    print("="*70)
    print("V8: Region-Specific Matching")
    print(f"Fixed clusters: {N_CLUSTERS} per sample")
    print("="*70)
    
    matcher = RegionMatcher(output_dir='./gene_to_mz_results_v8')
    matcher.load_all_data()
    results = matcher.run_analysis(epochs=150, top_k=20)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    
    return matcher, results


In [19]:
if __name__ == "__main__":
    matcher, results = main()

V8: Region-Specific Matching
Fixed clusters: 3 per sample
Loading RNA-seq data...
  Pixel size: 55 μm (hexagonal)

Loading MSI data...
  Pixel size: 60 μm (Cartesian)
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

GENE-TO-M/Z MATCHING V8
Region-Specific Comparison with Leiden Clustering
Fixed clusters per sample: 3

Computing Leiden clusters (target: 3 clusters per sample)...

RNA clusters:

MSI clusters:
  YC_1: 3 clusters
  YC_2: 3 clusters
  YC_3: 3 clusters
  YC_4: 3 clusters
  YAD_1: 3 clusters
  YAD_2: 3 clusters
  YAD_3: 3 clusters
  YAD_4: 3 clusters
  AC_1: 3 clusters
  AC_2: 3 clusters
  AC_3: 3 clusters
  AC_4: 3 clusters
  AAD_1: 3 clusters
  AAD_2: 3 clusters
  AAD_3: 3 clusters
  AAD_4: 3 cluste

ValueError: No RNA data available for training!